# Drive → BigQuery

Loads your Google Drive into BigQuery project `pelagic-gist-505800-b9` as three datasets:

| Dataset | Contents |
|---|---|
| `drive_raw.file_manifest` | One row per file in Drive — the census. Every file appears whether it loaded or not, with its error. Nothing disappears silently. |
| `drive_tables.<name>` | One table per *family* of tabular files. |
| `drive_documents.documents` | Extracted text from PDF, Word, PowerPoint, txt, md. |

**Why this exists.** A Takeout/Fitbit export is thousands of files that are really a few dozen tables sharded by day:

```
heart_rate_2026-03-23.csv
heart_rate_2026-03-24.csv
heart_rate_2026-04-05.csv
```

Loading one table per file gives you 1000+ useless single-day tables. These are grouped into one `heart_rate` table, with the date kept as a `_src_date` column. On the first 100 CSVs in this Drive: **100 files → 28 tables.**

**Why Colab.** Your Drive mounts as ordinary files, so there is no API pagination and no download quota, and BigQuery authenticates as *you* — no service account, no JSON key, nothing to paste.

Run the cells in order. Nothing is written to BigQuery until step 5.

## 1. Install dependencies

In [ ]:
%pip install --quiet google-cloud-bigquery google-api-python-client \
    pandas pyarrow openpyxl xlrd pypdf python-docx python-pptx
print('dependencies installed')

## 2. Write out the pipeline

In [ ]:
# The pipeline modules, embedded so this notebook needs no clone step.
# Generated by build_notebook.py -- edit the .py files, not this cell.
import pathlib, textwrap

MODULES = {}

MODULES['classify.py'] = r'''__CLASSIFY_PY__
"""Filename/MIME classification and table-family grouping.

The central idea: a Google Takeout / Fitbit export contains thousands of files
that are really *one table each, sharded by date*. `heart_rate_2026-04-05.csv`
and `heart_rate_2026-07-05.csv` are not two tables, they are two days of one
table. Grouping them into "families" is what turns 1000+ loose files into a few
dozen queryable tables.
"""

from __future__ import annotations

import posixpath
import re
from dataclasses import dataclass, field

# MIME types we can parse into rows.
TABULAR_MIMES = {
    "text/csv": "csv",
    "text/tab-separated-values": "tsv",
    "application/vnd.ms-excel": "xls",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": "xlsx",
    "application/vnd.google-apps.spreadsheet": "gsheet",
    "application/json": "json",
    "application/x-ndjson": "ndjson",
}

# MIME types we extract text from.
DOCUMENT_MIMES = {
    "application/pdf": "pdf",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": "docx",
    "application/msword": "doc",
    "application/vnd.google-apps.document": "gdoc",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": "pptx",
    "application/vnd.google-apps.presentation": "gslides",
    "text/plain": "txt",
    "text/markdown": "md",
    "text/html": "html",
    "application/rtf": "rtf",
}

# Extensions win when Drive reports a useless MIME type (very common for
# Takeout uploads, which often arrive as application/octet-stream).
EXTENSION_OVERRIDES = {
    "csv": ("tabular", "csv"),
    "tsv": ("tabular", "tsv"),
    "xlsx": ("tabular", "xlsx"),
    "xlsm": ("tabular", "xlsx"),
    "xls": ("tabular", "xls"),
    "json": ("tabular", "json"),
    "ndjson": ("tabular", "ndjson"),
    "jsonl": ("tabular", "ndjson"),
    "pdf": ("document", "pdf"),
    "docx": ("document", "docx"),
    "doc": ("document", "doc"),
    "pptx": ("document", "pptx"),
    "txt": ("document", "txt"),
    "md": ("document", "md"),
    "html": ("document", "html"),
    "htm": ("document", "html"),
    "rtf": ("document", "rtf"),
}

# A mounted Drive reports no MIME type, so media has to be recognised by
# extension or it all lands in the "other" bucket.
MEDIA_EXTENSIONS = {
    "jpg", "jpeg", "png", "gif", "bmp", "tif", "tiff", "webp", "heic", "heif",
    "svg", "ico", "raw", "cr2", "nef",
    "mp4", "mov", "avi", "mkv", "webm", "wmv", "flv", "m4v", "mpg", "mpeg",
    "mp3", "wav", "aac", "flac", "ogg", "m4a", "wma", "aiff",
    "ttf", "otf", "woff", "woff2", "eot",
}

FOLDER_MIME = "application/vnd.google-apps.folder"

MEDIA_PREFIXES = ("image/", "video/", "audio/", "font/")

# Trailing shard tokens stripped to find the family stem, longest-first so that
# `_2026-05-01` is consumed before the bare `_2026` rule can nibble at it.
_SHARD_PATTERNS = [
    r"[ _-]+\d{4}[-_]\d{2}[-_]\d{2}(?:[ _-]?\d{2}[-_:]\d{2}(?:[-_:]\d{2})?)?$",
    r"[ _-]+\d{4}[-_]\d{2}$",
    r"[ _-]+\d{8}$",
    r"[ _-]+\d{4}$",
    r"\s*\((\d+)\)$",
    r"[ _-]+copy$",
    r"[ _-]+final$",
    r"[ _-]+v\d+$",
    r"[ _-]+part[ _-]?\d+$",
    r"[ _-]+\d+of\d+$",
]

_DATE_IN_NAME = re.compile(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})")


def classify(name: str, mime_type: str) -> tuple[str, str]:
    """Return ``(kind, fmt)`` where kind is tabular/document/media/other/folder."""
    if mime_type == FOLDER_MIME:
        return "folder", "folder"

    ext = extension_of(name)
    # Drive's MIME is authoritative for native Google types; otherwise the
    # extension is more trustworthy than octet-stream.
    if mime_type.startswith("application/vnd.google-apps."):
        if mime_type in TABULAR_MIMES:
            return "tabular", TABULAR_MIMES[mime_type]
        if mime_type in DOCUMENT_MIMES:
            return "document", DOCUMENT_MIMES[mime_type]
        return "other", mime_type.rsplit(".", 1)[-1]

    if ext in EXTENSION_OVERRIDES:
        return EXTENSION_OVERRIDES[ext]
    if mime_type in TABULAR_MIMES:
        return "tabular", TABULAR_MIMES[mime_type]
    if mime_type in DOCUMENT_MIMES:
        return "document", DOCUMENT_MIMES[mime_type]
    if mime_type.startswith(MEDIA_PREFIXES):
        return "media", mime_type.split("/", 1)[0]
    if ext in MEDIA_EXTENSIONS:
        return "media", ext
    return "other", ext or "unknown"


def extension_of(name: str) -> str:
    base = posixpath.basename(name)
    if "." not in base:
        return ""
    return base.rsplit(".", 1)[-1].lower()


def strip_extension(name: str) -> str:
    base = posixpath.basename(name)
    if "." in base and extension_of(base):
        return base.rsplit(".", 1)[0]
    return base


def family_stem(name: str) -> str:
    """Strip date/version shard suffixes to get the shared stem of a family."""
    stem = strip_extension(name)
    changed = True
    while changed:
        changed = False
        for pattern in _SHARD_PATTERNS:
            new = re.sub(pattern, "", stem, flags=re.IGNORECASE)
            if new != stem and new.strip(" _-"):
                stem = new
                changed = True
    return stem.strip(" _-")


def shard_date(name: str) -> str | None:
    """Best-effort ISO date pulled from a filename, for the ``_src_date`` column."""
    match = _DATE_IN_NAME.search(strip_extension(name))
    if not match:
        return None
    year, month, day = match.groups()
    if not (1970 <= int(year) <= 2100):
        return None
    if not (1 <= int(month) <= 12 and 1 <= int(day) <= 31):
        return None
    return f"{year}-{month}-{day}"


def sanitize_table_name(stem: str, fallback: str = "unnamed") -> str:
    """Coerce an arbitrary filename stem into a legal BigQuery table id."""
    name = re.sub(r"[^0-9a-zA-Z_]+", "_", stem).strip("_").lower()
    name = re.sub(r"_{2,}", "_", name)
    if not name:
        name = fallback
    if name[0].isdigit():
        name = f"t_{name}"
    return name[:1024]


def sanitize_column_name(raw: str, position: int) -> str:
    """Coerce a CSV header cell into a legal BigQuery column id."""
    name = re.sub(r"[^0-9a-zA-Z_]+", "_", str(raw)).strip("_").lower()
    name = re.sub(r"_{2,}", "_", name)
    if not name:
        name = f"col_{position}"
    if name[0].isdigit() or name.startswith("_"):
        name = f"c_{name.lstrip('_')}"
    # BigQuery reserves the _TABLE_/_FILE_/_PARTITION prefixes.
    for reserved in ("_table_", "_file_", "_partition"):
        if name.startswith(reserved):
            name = f"c{name}"
    return name[:300]


@dataclass
class Family:
    """A set of Drive files that should land in one BigQuery table."""

    table: str
    stem: str
    fmt: str
    files: list[dict] = field(default_factory=list)

    @property
    def total_bytes(self) -> int:
        return sum(int(f.get("size_bytes") or 0) for f in self.files)


def build_families(files: list[dict]) -> dict[str, Family]:
    """Group classified tabular file records into families keyed by table name.

    Files sharing a stem but differing in format stay separate, because a CSV
    and an XLSX of the same stem rarely share a schema.
    """
    families: dict[str, Family] = {}
    for record in files:
        if record.get("kind") != "tabular":
            continue
        stem = family_stem(record["name"])
        fmt = record.get("fmt") or extension_of(record["name"])
        table = sanitize_table_name(stem)
        # Spreadsheets and CSVs of the same stem get distinct tables.
        if fmt in {"xlsx", "xls", "gsheet"}:
            table = f"{table}_sheet"
        elif fmt in {"json", "ndjson"}:
            table = f"{table}_json"
        key = table
        if key not in families:
            families[key] = Family(table=table, stem=stem, fmt=fmt)
        families[key].files.append(record)
    return families
__CLASSIFY_PY__'''

MODULES['drive.py'] = r'''__DRIVE_PY__
"""Drive enumeration and download.

Two modes, yielding identically shaped records so everything downstream is
mode-agnostic:

* ``walk`` / ``download`` -- the Drive API, for a service account or ADC.
* ``walk_local`` / ``read_local`` -- an already-mounted Drive (Colab's
  ``drive.mount``, or Drive for Desktop). Files are ordinary paths, so there is
  no pagination, no per-file API call, and no download quota. This is by far the
  cheaper mode for a Drive with thousands of files.
"""

from __future__ import annotations

import datetime as dt
import io
import json
import os
import time
from pathlib import Path

from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload

from classify import FOLDER_MIME, classify, extension_of, family_stem, shard_date

FIELDS = (
    "nextPageToken,files(id,name,mimeType,size,md5Checksum,createdTime,"
    "modifiedTime,parents,trashed,shortcutDetails)"
)

# Google-native formats have no bytes to download; they must be exported.
EXPORT_MIMES = {
    "application/vnd.google-apps.spreadsheet": (
        "text/csv",
        "csv",
    ),
    "application/vnd.google-apps.document": (
        "text/plain",
        "txt",
    ),
    "application/vnd.google-apps.presentation": (
        "text/plain",
        "txt",
    ),
}

RETRYABLE_STATUS = {403, 429, 500, 502, 503, 504}


def _retry(call, attempts: int = 5, what: str = "drive call"):
    """Retry a Drive request through rate limits with exponential backoff."""
    delay = 2.0
    last: Exception | None = None
    for attempt in range(attempts):
        try:
            return call()
        except HttpError as exc:  # pragma: no cover - network dependent
            status = getattr(exc.resp, "status", None)
            if status not in RETRYABLE_STATUS or attempt == attempts - 1:
                raise
            last = exc
            time.sleep(delay)
            delay *= 2
    raise RuntimeError(f"{what} failed after {attempts} attempts: {last}")


def walk(service, root_id: str | None = None, include_trashed: bool = False):
    """Yield file records under ``root_id`` (whole My Drive when None).

    Breadth-first so that a partial run still covers whole folders, and so the
    path column can be built incrementally without a second lookup.
    """
    if root_id:
        root_meta = _retry(
            lambda: service.files()
            .get(fileId=root_id, fields="id,name,mimeType", supportsAllDrives=True)
            .execute(),
            what=f"get root {root_id}",
        )
        queue = [(root_id, root_meta.get("name", "/"))]
        seen_folders = {root_id}
    else:
        queue = [("root", "")]
        seen_folders = {"root"}

    while queue:
        folder_id, folder_path = queue.pop(0)
        page_token = None
        while True:
            query = f"'{folder_id}' in parents"
            if not include_trashed:
                query += " and trashed = false"

            def _list(token=page_token, q=query):
                return (
                    service.files()
                    .list(
                        q=q,
                        fields=FIELDS,
                        pageSize=1000,
                        pageToken=token,
                        supportsAllDrives=True,
                        includeItemsFromAllDrives=True,
                    )
                    .execute()
                )

            response = _retry(_list, what=f"list {folder_id}")

            for item in response.get("files", []):
                name = item.get("name", "")
                mime = item.get("mimeType", "")
                path = f"{folder_path}/{name}" if folder_path else name

                if mime == FOLDER_MIME:
                    if item["id"] not in seen_folders:
                        seen_folders.add(item["id"])
                        queue.append((item["id"], path))
                    continue

                # Shortcuts point elsewhere; the target is enumerated on its own.
                if mime == "application/vnd.google-apps.shortcut":
                    continue

                kind, fmt = classify(name, mime)
                yield {
                    "file_id": item["id"],
                    "name": name,
                    "path": path,
                    "mime_type": mime,
                    "extension": extension_of(name),
                    "size_bytes": int(item["size"]) if item.get("size") else None,
                    "md5_checksum": item.get("md5Checksum"),
                    "created_time": item.get("createdTime"),
                    "modified_time": item.get("modifiedTime"),
                    "parent_id": (item.get("parents") or [None])[0],
                    "kind": kind,
                    "fmt": fmt,
                    "family_stem": family_stem(name) if kind == "tabular" else None,
                    "shard_date": shard_date(name),
                }

            page_token = response.get("nextPageToken")
            if not page_token:
                break


# A mounted Drive represents Google-native files as small JSON stub files with
# these extensions. The stub holds the real file id but none of the data, so the
# bytes still have to be exported through the API.
STUB_EXTENSIONS = {
    "gsheet": "application/vnd.google-apps.spreadsheet",
    "gdoc": "application/vnd.google-apps.document",
    "gslides": "application/vnd.google-apps.presentation",
    "gdraw": "application/vnd.google-apps.drawing",
    "gform": "application/vnd.google-apps.form",
}

# Mount bookkeeping that is not user data.
SKIP_NAMES = {".shortcut-targets-by-id", ".file-revisions-by-id", ".Trash", ".DS_Store"}


def _stub_file_id(path: Path) -> str | None:
    """Pull the Drive file id out of a mounted .gsheet/.gdoc stub."""
    try:
        payload = json.loads(path.read_text(encoding="utf-8", errors="replace"))
    except (OSError, json.JSONDecodeError):
        return None
    # Stubs carry either {"doc_id": ...} or {"url": "...open?id=FILE_ID"}.
    if isinstance(payload, dict):
        if payload.get("doc_id"):
            return str(payload["doc_id"])
        url = payload.get("url") or ""
        if "id=" in url:
            return url.split("id=", 1)[1].split("&", 1)[0]
    return None


def walk_local(root: str, include_hidden: bool = False):
    """Yield file records from a mounted Drive directory tree.

    Native Google files are surfaced with their real Drive id and MIME type so
    that ``read_local`` can export them through the API; everything else is read
    straight off disk.
    """
    root_path = Path(root).expanduser().resolve()
    if not root_path.is_dir():
        raise NotADirectoryError(f"{root_path} is not a directory")

    for dirpath, dirnames, filenames in os.walk(root_path):
        # Prune in place so os.walk does not descend into them.
        dirnames[:] = [
            d
            for d in dirnames
            if d not in SKIP_NAMES and (include_hidden or not d.startswith("."))
        ]
        for filename in filenames:
            if filename in SKIP_NAMES:
                continue
            if not include_hidden and filename.startswith("."):
                continue

            full = Path(dirpath) / filename
            try:
                stat = full.stat()
            except OSError:
                continue

            relative = full.relative_to(root_path).as_posix()
            ext = extension_of(filename)

            if ext in STUB_EXTENSIONS:
                mime = STUB_EXTENSIONS[ext]
                file_id = _stub_file_id(full) or f"local:{relative}"
                size = None  # the stub's size is meaningless
            else:
                mime = ""
                file_id = f"local:{relative}"
                size = stat.st_size

            kind, fmt = classify(filename, mime)
            yield {
                "file_id": file_id,
                "name": filename,
                "path": relative,
                "local_path": str(full),
                "mime_type": mime,
                "extension": ext,
                "size_bytes": size,
                "md5_checksum": None,
                "created_time": _iso(stat.st_ctime),
                "modified_time": _iso(stat.st_mtime),
                "parent_id": Path(dirpath).name,
                "kind": kind,
                "fmt": fmt,
                "family_stem": family_stem(filename) if kind == "tabular" else None,
                "shard_date": shard_date(filename),
            }


def _iso(timestamp: float) -> str:
    return dt.datetime.fromtimestamp(timestamp, dt.timezone.utc).isoformat()


def read_local(record: dict, service=None) -> tuple[bytes, str]:
    """Read a mounted file's bytes, exporting native Google files via the API.

    ``service`` is only needed for native files; plain files never touch the API.
    """
    mime = record.get("mime_type") or ""
    if mime in EXPORT_MIMES:
        if service is None:
            raise RuntimeError(
                f"{record['name']} is a native Google file and needs a Drive "
                "service to export; pass credentials or skip it"
            )
        return download(service, record["file_id"], mime)
    return Path(record["local_path"]).read_bytes(), ""


def download(service, file_id: str, mime_type: str) -> tuple[bytes, str]:
    """Fetch a file's bytes. Returns ``(data, effective_format)``.

    Google-native files are exported; everything else is downloaded verbatim.
    """
    if mime_type in EXPORT_MIMES:
        export_mime, fmt = EXPORT_MIMES[mime_type]
        request = service.files().export_media(fileId=file_id, mimeType=export_mime)
    else:
        fmt = ""
        request = service.files().get_media(fileId=file_id, supportsAllDrives=True)

    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request, chunksize=8 * 1024 * 1024)
    done = False
    while not done:
        _, done = _retry(downloader.next_chunk, what=f"download {file_id}")
    return buffer.getvalue(), fmt
__DRIVE_PY__'''

MODULES['parse.py'] = r'''__PARSE_PY__
"""Turn downloaded bytes into DataFrames, and documents into text."""

from __future__ import annotations

import io
import json

import pandas as pd

from classify import sanitize_column_name

# Read everything as string first, then let BigQuery/pandas infer on the merged
# frame. Per-file inference is what causes schema drift across a family (one
# day's file has all-integer values, the next has a decimal).
_READ_KWARGS = {"dtype": str, "keep_default_na": False, "na_values": [""]}

MAX_TEXT_CHARS = 900_000  # keep a row comfortably under BigQuery's 100 MB cap


def normalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Sanitize headers and de-duplicate collisions."""
    seen: dict[str, int] = {}
    columns = []
    for position, raw in enumerate(frame.columns):
        name = sanitize_column_name(raw, position)
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        columns.append(name)
    frame.columns = columns
    return frame


def read_tabular(data: bytes, fmt: str, name: str) -> list[tuple[str, pd.DataFrame]]:
    """Parse bytes into ``[(sheet_suffix, frame)]``.

    Multi-sheet workbooks yield one entry per sheet; everything else yields one
    entry with an empty suffix.
    """
    if fmt in {"csv", "tsv"}:
        sep = "\t" if fmt == "tsv" else ","
        frame = _read_csv(data, sep)
        return [("", normalize_columns(frame))]

    if fmt in {"xlsx", "xls", "xlsm"}:
        engine = "openpyxl" if fmt != "xls" else "xlrd"
        book = pd.read_excel(
            io.BytesIO(data), sheet_name=None, engine=engine, **_READ_KWARGS
        )
        out = []
        for sheet_name, frame in book.items():
            if frame.empty:
                continue
            out.append((str(sheet_name), normalize_columns(frame)))
        return out

    if fmt in {"json", "ndjson"}:
        return [("", normalize_columns(_read_json(data)))]

    raise ValueError(f"unsupported tabular format {fmt!r} for {name!r}")


def _read_csv(data: bytes, sep: str) -> pd.DataFrame:
    """Read a CSV, tolerating the encodings Takeout exports show up in."""
    last: Exception | None = None
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            return pd.read_csv(
                io.BytesIO(data),
                sep=sep,
                encoding=encoding,
                engine="python",
                on_bad_lines="skip",
                **_READ_KWARGS,
            )
        except (UnicodeDecodeError, pd.errors.ParserError) as exc:
            last = exc
    raise ValueError(f"could not parse CSV: {last}")


def _read_json(data: bytes) -> pd.DataFrame:
    """Flatten JSON into a frame, handling arrays, NDJSON, and wrapped objects."""
    text = data.decode("utf-8", errors="replace").strip()
    if not text:
        return pd.DataFrame()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        # Assume NDJSON.
        rows = []
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
        return pd.json_normalize(rows) if rows else pd.DataFrame()

    if isinstance(parsed, list):
        return pd.json_normalize(parsed)
    if isinstance(parsed, dict):
        # Takeout wraps the payload in a single key more often than not
        # (e.g. {"features": [...]} in Saved Places).
        list_values = [v for v in parsed.values() if isinstance(v, list)]
        if len(list_values) == 1:
            return pd.json_normalize(list_values[0])
        return pd.json_normalize([parsed])
    return pd.DataFrame({"value": [parsed]})


def extract_text(data: bytes, fmt: str) -> tuple[str, int, str]:
    """Return ``(text, page_count, method)`` for a document file."""
    if fmt in {"txt", "md", "html", "rtf"}:
        text = data.decode("utf-8", errors="replace")
        if fmt == "html":
            text = _strip_html(text)
        return text[:MAX_TEXT_CHARS], 0, f"decode:{fmt}"

    if fmt == "pdf":
        try:
            from pypdf import PdfReader
        except ImportError:
            return "", 0, "skipped:pypdf-missing"
        reader = PdfReader(io.BytesIO(data))
        pages = [(page.extract_text() or "") for page in reader.pages]
        return "\n".join(pages)[:MAX_TEXT_CHARS], len(pages), "pypdf"

    if fmt == "docx":
        try:
            import docx
        except ImportError:
            return "", 0, "skipped:python-docx-missing"
        document = docx.Document(io.BytesIO(data))
        text = "\n".join(p.text for p in document.paragraphs)
        return text[:MAX_TEXT_CHARS], 0, "python-docx"

    if fmt == "pptx":
        try:
            from pptx import Presentation
        except ImportError:
            return "", 0, "skipped:python-pptx-missing"
        deck = Presentation(io.BytesIO(data))
        chunks = []
        for slide in deck.slides:
            for shape in slide.shapes:
                if getattr(shape, "has_text_frame", False):
                    chunks.append(shape.text_frame.text)
        return "\n".join(chunks)[:MAX_TEXT_CHARS], len(deck.slides), "python-pptx"

    return "", 0, f"unsupported:{fmt}"


def _strip_html(text: str) -> str:
    import re

    text = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", text, flags=re.S | re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s{2,}", " ", text).strip()


def reconcile(frames: list[pd.DataFrame]) -> pd.DataFrame:
    """Union frames with differing columns into one, preserving every column.

    Missing columns become NULL rather than dropping the row — a day's file that
    lacks a column should not silently lose its other fields.
    """
    if not frames:
        return pd.DataFrame()
    ordered: list[str] = []
    for frame in frames:
        for column in frame.columns:
            if column not in ordered:
                ordered.append(column)
    aligned = [frame.reindex(columns=ordered) for frame in frames]
    return pd.concat(aligned, ignore_index=True)
__PARSE_PY__'''

MODULES['bq.py'] = r'''__BQ_PY__
"""BigQuery dataset/table management and loading."""

from __future__ import annotations

import io
import logging

import pandas as pd
from google.api_core import exceptions as gexc
from google.cloud import bigquery

log = logging.getLogger(__name__)

MANIFEST_TABLE = "file_manifest"
DOCUMENTS_TABLE = "documents"

MANIFEST_SCHEMA = [
    bigquery.SchemaField("file_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("path", "STRING"),
    bigquery.SchemaField("mime_type", "STRING"),
    bigquery.SchemaField("extension", "STRING"),
    bigquery.SchemaField("size_bytes", "INT64"),
    bigquery.SchemaField("md5_checksum", "STRING"),
    bigquery.SchemaField("created_time", "TIMESTAMP"),
    bigquery.SchemaField("modified_time", "TIMESTAMP"),
    bigquery.SchemaField("parent_id", "STRING"),
    bigquery.SchemaField("kind", "STRING"),
    bigquery.SchemaField("fmt", "STRING"),
    bigquery.SchemaField("family_stem", "STRING"),
    bigquery.SchemaField("shard_date", "DATE"),
    bigquery.SchemaField("target_dataset", "STRING"),
    bigquery.SchemaField("target_table", "STRING"),
    bigquery.SchemaField("ingest_status", "STRING"),
    bigquery.SchemaField("ingest_error", "STRING"),
    bigquery.SchemaField("row_count", "INT64"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]

DOCUMENTS_SCHEMA = [
    bigquery.SchemaField("file_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("path", "STRING"),
    bigquery.SchemaField("mime_type", "STRING"),
    bigquery.SchemaField("fmt", "STRING"),
    bigquery.SchemaField("size_bytes", "INT64"),
    bigquery.SchemaField("created_time", "TIMESTAMP"),
    bigquery.SchemaField("modified_time", "TIMESTAMP"),
    bigquery.SchemaField("page_count", "INT64"),
    bigquery.SchemaField("char_count", "INT64"),
    bigquery.SchemaField("extraction_method", "STRING"),
    bigquery.SchemaField("content", "STRING"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]

# Provenance columns prepended to every table in the tables dataset, so a row
# can always be traced back to the Drive file it came from.
PROVENANCE = ("_src_file_id", "_src_file_name", "_src_date", "_src_sheet", "_ingested_at")


class Loader:
    def __init__(self, project: str, location: str = "US", dry_run: bool = False):
        self.project = project
        self.location = location
        self.dry_run = dry_run
        self.client = None if dry_run else bigquery.Client(project=project)

    # ---------------------------------------------------------------- datasets

    def ensure_dataset(self, dataset_id: str) -> None:
        if self.dry_run:
            log.info("[dry-run] ensure dataset %s.%s", self.project, dataset_id)
            return
        ref = bigquery.Dataset(f"{self.project}.{dataset_id}")
        ref.location = self.location
        try:
            self.client.create_dataset(ref)
            log.info("created dataset %s", dataset_id)
        except gexc.Conflict:
            log.debug("dataset %s already exists", dataset_id)

    def ensure_table(self, dataset_id: str, table_id: str, schema: list) -> None:
        if self.dry_run:
            log.info("[dry-run] ensure table %s.%s", dataset_id, table_id)
            return
        table = bigquery.Table(f"{self.project}.{dataset_id}.{table_id}", schema=schema)
        try:
            self.client.create_table(table)
            log.info("created table %s.%s", dataset_id, table_id)
        except gexc.Conflict:
            log.debug("table %s.%s already exists", dataset_id, table_id)

    # ------------------------------------------------------------------ loads

    def load_frame(
        self,
        frame: pd.DataFrame,
        dataset_id: str,
        table_id: str,
        write_disposition: str = "WRITE_APPEND",
    ) -> int:
        """Load a DataFrame via Parquet, letting the schema widen as needed."""
        if frame.empty:
            return 0
        if self.dry_run:
            log.info(
                "[dry-run] load %d rows x %d cols -> %s.%s",
                len(frame),
                len(frame.columns),
                dataset_id,
                table_id,
            )
            return len(frame)

        buffer = io.BytesIO()
        frame.to_parquet(buffer, index=False, engine="pyarrow")
        buffer.seek(0)

        config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.PARQUET,
            write_disposition=write_disposition,
            autodetect=True,
            schema_update_options=[
                bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION,
                bigquery.SchemaUpdateOption.ALLOW_FIELD_RELAXATION,
            ],
        )
        target = f"{self.project}.{dataset_id}.{table_id}"
        job = self.client.load_table_from_file(buffer, target, job_config=config)
        job.result()
        if job.errors:
            raise RuntimeError(f"load into {target} failed: {job.errors}")
        return len(frame)

    def load_rows(self, rows: list[dict], dataset_id: str, table_id: str, schema: list) -> int:
        """Load explicit dict rows against a fixed schema (manifest, documents)."""
        if not rows:
            return 0
        if self.dry_run:
            log.info("[dry-run] load %d rows -> %s.%s", len(rows), dataset_id, table_id)
            return len(rows)

        frame = pd.DataFrame(rows)
        buffer = io.BytesIO()
        frame.to_parquet(buffer, index=False, engine="pyarrow")
        buffer.seek(0)
        config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.PARQUET,
            write_disposition="WRITE_APPEND",
            schema=schema,
        )
        target = f"{self.project}.{dataset_id}.{table_id}"
        job = self.client.load_table_from_file(buffer, target, job_config=config)
        job.result()
        if job.errors:
            raise RuntimeError(f"load into {target} failed: {job.errors}")
        return len(rows)

    # ----------------------------------------------------------------- resume

    def already_ingested(self, dataset_id: str) -> set[str]:
        """File ids already recorded as loaded, so a rerun resumes cleanly."""
        if self.dry_run:
            return set()
        query = f"""
            SELECT DISTINCT file_id
            FROM `{self.project}.{dataset_id}.{MANIFEST_TABLE}`
            WHERE ingest_status = 'loaded'
        """
        try:
            return {row.file_id for row in self.client.query(query).result()}
        except gexc.NotFound:
            return set()
__BQ_PY__'''

MODULES['pipeline.py'] = r'''__PIPELINE_PY__
#!/usr/bin/env python3
"""Load an entire Google Drive into BigQuery.

Three datasets, three jobs:

  drive_raw        file_manifest -- one row per file in Drive, loaded or not.
                   The census. Nothing is silently dropped; anything that fails
                   is recorded here with its error.
  drive_tables     one table per *family* of tabular files. Date-sharded exports
                   (heart_rate_2026-04-05.csv, heart_rate_2026-07-05.csv, ...)
                   collapse into one table with _src_date provenance.
  drive_documents  documents -- extracted text from PDFs, Word, slides, txt/md.

Usage
-----
    python pipeline.py inventory --project PROJECT [--folder FOLDER_ID]
    python pipeline.py plan      --project PROJECT [--folder FOLDER_ID]
    python pipeline.py load      --project PROJECT [--folder FOLDER_ID]

`inventory` writes only the manifest. `plan` prints the table layout without
touching BigQuery. `load` does everything and is safe to re-run: files already
marked loaded in the manifest are skipped.
"""

from __future__ import annotations

import argparse
import datetime as dt
import json
import logging
import os
import sys
import traceback
from pathlib import Path

import pandas as pd
from google.auth import default as google_auth_default
from google.oauth2 import service_account
from googleapiclient.discovery import build

sys.path.insert(0, str(Path(__file__).resolve().parent))

import bq  # noqa: E402
import drive as drive_mod  # noqa: E402
from classify import build_families, sanitize_table_name  # noqa: E402
from parse import extract_text, read_tabular, reconcile  # noqa: E402

SCOPES = [
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/bigquery",
]

RAW_DATASET = "drive_raw"
TABLES_DATASET = "drive_tables"
DOCS_DATASET = "drive_documents"

# Files bigger than this are recorded in the manifest but not parsed, so one
# 4 GB video cannot stall a run over thousands of small files.
MAX_PARSE_BYTES = 512 * 1024 * 1024

# Rows are accumulated per family and flushed in batches to bound memory.
FLUSH_ROWS = 400_000

log = logging.getLogger("drive2bq")


# --------------------------------------------------------------------- helpers


def credentials():
    """Service-account JSON if provided, otherwise Application Default Creds."""
    key_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if key_path and Path(key_path).is_file():
        return service_account.Credentials.from_service_account_file(
            key_path, scopes=SCOPES
        )
    creds, _ = google_auth_default(scopes=SCOPES)
    return creds


def now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def _is_stringish(series: pd.Series) -> bool:
    """True for object and for pandas' native string dtypes.

    pandas 2.2 gives ``object``, pandas 3 gives ``str``; checking only for
    ``object`` silently skips every column on newer pandas and leaves the whole
    table as strings.
    """
    return series.dtype == object or pd.api.types.is_string_dtype(series.dtype)


def coerce_types(frame: pd.DataFrame) -> pd.DataFrame:
    """Promote all-string columns to numeric/timestamp where unambiguous.

    Everything is read as string to keep a family's files schema-compatible;
    this puts the real types back once the whole family is merged, so the
    resulting tables are actually queryable with SUM/AVG and date filters.
    """
    for column in frame.columns:
        if column.startswith("_src_") or column == "_ingested_at":
            continue
        series = frame[column]
        if not _is_stringish(series):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue

        numeric = pd.to_numeric(non_null, errors="coerce")
        if numeric.notna().all():
            full = pd.to_numeric(series, errors="coerce")
            # Keep integers narrow when there is no fractional part.
            if (full.dropna() % 1 == 0).all():
                frame[column] = full.astype("Int64")
            else:
                frame[column] = full
            continue

        # Only try dates on values that look like them, so free text that
        # happens to start with a digit is not mangled into 1970.
        sample = non_null.astype(str).head(200)
        if sample.str.match(r"^\d{4}-\d{2}-\d{2}([ T]|$)").mean() > 0.9:
            parsed = pd.to_datetime(series, errors="coerce", format="mixed", utc=True)
            if parsed.notna().sum() >= non_null.shape[0] * 0.99:
                frame[column] = parsed

    # Provenance columns get real types too, so _src_date is filterable as a
    # DATE and _ingested_at as a TIMESTAMP rather than both landing as strings.
    if "_src_date" in frame.columns:
        frame["_src_date"] = pd.to_datetime(
            frame["_src_date"], errors="coerce", format="%Y-%m-%d"
        ).dt.date
    if "_ingested_at" in frame.columns:
        frame["_ingested_at"] = pd.to_datetime(
            frame["_ingested_at"], errors="coerce", utc=True
        )
    return frame


def add_provenance(frame: pd.DataFrame, record: dict, sheet: str) -> pd.DataFrame:
    frame = frame.copy()
    frame.insert(0, "_src_file_id", record["file_id"])
    frame.insert(1, "_src_file_name", record["name"])
    frame.insert(2, "_src_date", record.get("shard_date"))
    frame.insert(3, "_src_sheet", sheet or None)
    frame.insert(4, "_ingested_at", now())
    return frame


def manifest_row(record: dict, **overrides) -> dict:
    row = {
        "file_id": record["file_id"],
        "name": record["name"],
        "path": record["path"],
        "mime_type": record["mime_type"],
        "extension": record["extension"],
        "size_bytes": record["size_bytes"],
        "md5_checksum": record["md5_checksum"],
        "created_time": record["created_time"],
        "modified_time": record["modified_time"],
        "parent_id": record["parent_id"],
        "kind": record["kind"],
        "fmt": record["fmt"],
        "family_stem": record["family_stem"],
        "shard_date": record["shard_date"],
        "target_dataset": None,
        "target_table": None,
        "ingest_status": "pending",
        "ingest_error": None,
        "row_count": None,
        "ingested_at": now(),
    }
    row.update(overrides)
    return row


def enumerate_drive(args, drive_service=None) -> list[dict]:
    """Enumerate via a mounted path when given, otherwise via the Drive API."""
    if args.local_root:
        log.info("enumerating mounted Drive at %s ...", args.local_root)
        files = list(drive_mod.walk_local(args.local_root))
    else:
        log.info(
            "enumerating Drive%s ...", f" folder {args.folder}" if args.folder else " (all)"
        )
        files = list(drive_mod.walk(drive_service, root_id=args.folder))
    log.info("found %d files", len(files))
    return files


def fetch_bytes(record: dict, args, drive_service) -> tuple[bytes, str]:
    """Read one file's bytes in whichever mode is active."""
    if args.local_root:
        return drive_mod.read_local(record, service=drive_service)
    return drive_mod.download(drive_service, record["file_id"], record["mime_type"])


def needs_api(files: list[dict]) -> bool:
    """True when any file can only be read by exporting through the API."""
    return any(f.get("mime_type") in drive_mod.EXPORT_MIMES for f in files)


def make_drive_service(optional: bool = False):
    """Build a Drive client, tolerating absent credentials in local mode."""
    try:
        return build("drive", "v3", credentials=credentials(), cache_discovery=False)
    except Exception as exc:
        if not optional:
            raise
        log.warning("no Drive credentials (%s); native Google files will be skipped", exc)
        return None


def summarize(files: list[dict]) -> dict:
    by_kind: dict[str, int] = {}
    bytes_by_kind: dict[str, int] = {}
    for record in files:
        by_kind[record["kind"]] = by_kind.get(record["kind"], 0) + 1
        bytes_by_kind[record["kind"]] = bytes_by_kind.get(record["kind"], 0) + int(
            record["size_bytes"] or 0
        )
    return {"count": len(files), "by_kind": by_kind, "bytes_by_kind": bytes_by_kind}


# ------------------------------------------------------------------- commands


def cmd_inventory(args) -> int:
    drive_service = make_drive_service(optional=bool(args.local_root))
    files = enumerate_drive(args, drive_service)

    stats = summarize(files)
    families = build_families(files)
    print(json.dumps({**stats, "families": len(families)}, indent=2))

    loader = bq.Loader(args.project, args.location, dry_run=args.dry_run)
    loader.ensure_dataset(RAW_DATASET)
    loader.ensure_table(RAW_DATASET, bq.MANIFEST_TABLE, bq.MANIFEST_SCHEMA)
    rows = [manifest_row(r, ingest_status="inventoried") for r in files]
    loader.load_rows(rows, RAW_DATASET, bq.MANIFEST_TABLE, bq.MANIFEST_SCHEMA)
    log.info("manifest written: %d rows", len(rows))

    Path(args.out).write_text(json.dumps(files, indent=2))
    log.info("local inventory cached at %s", args.out)
    return 0


def cmd_plan(args) -> int:
    # Planning from a cached inventory needs no credentials at all, which makes
    # the table layout reviewable before any access is granted.
    if args.from_cache and Path(args.out).is_file():
        files = json.loads(Path(args.out).read_text())
        log.info("loaded %d files from cache %s", len(files), args.out)
    else:
        # A mounted walk needs no credentials at all; the API walk does.
        drive_service = make_drive_service(optional=bool(args.local_root))
        files = enumerate_drive(args, drive_service)
        Path(args.out).write_text(json.dumps(files, indent=2))

    stats = summarize(files)
    families = build_families(files)

    print(f"\nFiles: {stats['count']}")
    for kind, count in sorted(stats["by_kind"].items(), key=lambda kv: -kv[1]):
        mib = stats["bytes_by_kind"].get(kind, 0) / 1024 / 1024
        print(f"  {kind:<10} {count:>6}  ({mib:,.1f} MiB)")

    print(f"\n{TABLES_DATASET}: {len(families)} tables")
    for name, family in sorted(families.items(), key=lambda kv: -len(kv[1].files)):
        mib = family.total_bytes / 1024 / 1024
        print(f"  {name:<48} {len(family.files):>5} files  ({mib:,.1f} MiB)")

    docs = [f for f in files if f["kind"] == "document"]
    other = [f for f in files if f["kind"] in {"media", "other"}]
    print(f"\n{DOCS_DATASET}.documents: {len(docs)} files")
    print(f"{RAW_DATASET}.file_manifest: {stats['count']} files "
          f"({len(other)} recorded as metadata only)")
    return 0


def cmd_load(args) -> int:
    drive_service = make_drive_service(optional=bool(args.local_root))
    loader = bq.Loader(args.project, args.location, dry_run=args.dry_run)

    for dataset in (RAW_DATASET, TABLES_DATASET, DOCS_DATASET):
        loader.ensure_dataset(dataset)
    loader.ensure_table(RAW_DATASET, bq.MANIFEST_TABLE, bq.MANIFEST_SCHEMA)
    loader.ensure_table(DOCS_DATASET, bq.DOCUMENTS_TABLE, bq.DOCUMENTS_SCHEMA)

    if args.from_cache and Path(args.out).is_file():
        files = json.loads(Path(args.out).read_text())
        log.info("loaded %d files from cache %s", len(files), args.out)
    else:
        files = enumerate_drive(args, drive_service)
        Path(args.out).write_text(json.dumps(files, indent=2))

    if args.local_root and drive_service is None and needs_api(files):
        native = sum(1 for f in files if f.get("mime_type") in drive_mod.EXPORT_MIMES)
        log.warning(
            "%d native Google files (Sheets/Docs/Slides) cannot be read from the "
            "mount without credentials; they will be recorded as failed",
            native,
        )

    done = set() if args.no_resume else loader.already_ingested(RAW_DATASET)
    if done:
        log.info("resuming: %d files already loaded, skipping", len(done))

    families = build_families(files)
    manifest: list[dict] = []
    counters = {"loaded": 0, "skipped": 0, "failed": 0, "rows": 0}

    # ---- tabular families -> drive_tables ---------------------------------
    for table_name, family in sorted(families.items(), key=lambda kv: -len(kv[1].files)):
        pending = [f for f in family.files if f["file_id"] not in done]
        if not pending:
            continue
        log.info(
            "family %s: %d files (%d already loaded)",
            table_name,
            len(pending),
            len(family.files) - len(pending),
        )

        buffered: list[pd.DataFrame] = []
        buffered_rows = 0
        first_write = args.replace

        def flush(frames, replace):
            if not frames:
                return 0
            merged = coerce_types(reconcile(frames))
            disposition = "WRITE_TRUNCATE" if replace else "WRITE_APPEND"
            return loader.load_frame(merged, TABLES_DATASET, table_name, disposition)

        for record in pending:
            size = int(record["size_bytes"] or 0)
            if size > MAX_PARSE_BYTES:
                manifest.append(
                    manifest_row(
                        record,
                        ingest_status="skipped",
                        ingest_error=f"exceeds MAX_PARSE_BYTES ({size} bytes)",
                        target_dataset=TABLES_DATASET,
                        target_table=table_name,
                    )
                )
                counters["skipped"] += 1
                continue
            try:
                data, exported_fmt = fetch_bytes(record, args, drive_service)
                fmt = exported_fmt or record["fmt"]
                sheets = read_tabular(data, fmt, record["name"])
                rows_here = 0
                for sheet_name, frame in sheets:
                    if frame.empty:
                        continue
                    stamped = add_provenance(frame, record, sheet_name)
                    buffered.append(stamped)
                    buffered_rows += len(stamped)
                    rows_here += len(stamped)

                manifest.append(
                    manifest_row(
                        record,
                        ingest_status="loaded",
                        row_count=rows_here,
                        target_dataset=TABLES_DATASET,
                        target_table=table_name,
                    )
                )
                counters["loaded"] += 1
                counters["rows"] += rows_here
            except Exception as exc:  # keep going; record the failure
                log.warning("failed %s (%s): %s", record["name"], record["file_id"], exc)
                manifest.append(
                    manifest_row(
                        record,
                        ingest_status="failed",
                        ingest_error=f"{type(exc).__name__}: {exc}"[:1000],
                        target_dataset=TABLES_DATASET,
                        target_table=table_name,
                    )
                )
                counters["failed"] += 1

            if buffered_rows >= FLUSH_ROWS:
                flush(buffered, first_write)
                first_write = False
                buffered, buffered_rows = [], 0

        flush(buffered, first_write)

    # ---- documents -> drive_documents -------------------------------------
    doc_rows: list[dict] = []
    for record in files:
        if record["kind"] != "document" or record["file_id"] in done:
            continue
        size = int(record["size_bytes"] or 0)
        if size > MAX_PARSE_BYTES:
            manifest.append(
                manifest_row(record, ingest_status="skipped", ingest_error="too large")
            )
            counters["skipped"] += 1
            continue
        try:
            data, exported_fmt = fetch_bytes(record, args, drive_service)
            fmt = exported_fmt or record["fmt"]
            text, pages, method = extract_text(data, fmt)
            doc_rows.append(
                {
                    "file_id": record["file_id"],
                    "name": record["name"],
                    "path": record["path"],
                    "mime_type": record["mime_type"],
                    "fmt": fmt,
                    "size_bytes": record["size_bytes"],
                    "created_time": record["created_time"],
                    "modified_time": record["modified_time"],
                    "page_count": pages,
                    "char_count": len(text),
                    "extraction_method": method,
                    "content": text,
                    "ingested_at": now(),
                }
            )
            manifest.append(
                manifest_row(
                    record,
                    ingest_status="loaded",
                    row_count=1,
                    target_dataset=DOCS_DATASET,
                    target_table=bq.DOCUMENTS_TABLE,
                )
            )
            counters["loaded"] += 1
        except Exception as exc:
            log.warning("failed doc %s: %s", record["name"], exc)
            manifest.append(
                manifest_row(
                    record,
                    ingest_status="failed",
                    ingest_error=f"{type(exc).__name__}: {exc}"[:1000],
                )
            )
            counters["failed"] += 1

        if len(doc_rows) >= 500:
            loader.load_rows(doc_rows, DOCS_DATASET, bq.DOCUMENTS_TABLE, bq.DOCUMENTS_SCHEMA)
            doc_rows = []

    loader.load_rows(doc_rows, DOCS_DATASET, bq.DOCUMENTS_TABLE, bq.DOCUMENTS_SCHEMA)

    # ---- everything else: metadata only ------------------------------------
    for record in files:
        if record["kind"] in {"media", "other"} and record["file_id"] not in done:
            manifest.append(manifest_row(record, ingest_status="metadata_only"))

    loader.load_rows(manifest, RAW_DATASET, bq.MANIFEST_TABLE, bq.MANIFEST_SCHEMA)

    log.info(
        "done: %d loaded, %d skipped, %d failed, %d rows into %d tables",
        counters["loaded"],
        counters["skipped"],
        counters["failed"],
        counters["rows"],
        len(families),
    )
    print(json.dumps(counters, indent=2))
    return 1 if counters["failed"] and args.strict else 0


# ----------------------------------------------------------------------- main


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("command", choices=["inventory", "plan", "load"])
    parser.add_argument("--project", required=True, help="GCP project id")
    parser.add_argument("--folder", help="Drive folder id to limit the walk to")
    parser.add_argument(
        "--local-root",
        help="path to an already-mounted Drive (e.g. /content/drive/MyDrive) to "
        "read from the filesystem instead of the Drive API",
    )
    parser.add_argument("--location", default="US", help="BigQuery dataset location")
    parser.add_argument("--out", default="drive_inventory.json", help="inventory cache")
    parser.add_argument("--from-cache", action="store_true", help="reuse cached inventory")
    parser.add_argument("--dry-run", action="store_true", help="no BigQuery writes")
    parser.add_argument("--replace", action="store_true", help="truncate tables first")
    parser.add_argument("--no-resume", action="store_true", help="reload everything")
    parser.add_argument("--strict", action="store_true", help="exit 1 on any failure")
    parser.add_argument("-v", "--verbose", action="store_true")
    args = parser.parse_args(argv)

    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s %(levelname)-7s %(message)s",
    )

    handler = {"inventory": cmd_inventory, "plan": cmd_plan, "load": cmd_load}[args.command]
    try:
        return handler(args)
    except Exception:
        traceback.print_exc()
        return 2


if __name__ == "__main__":
    raise SystemExit(main())
__PIPELINE_PY__'''

for _name, _text in MODULES.items():
    # Strip the heredoc markers off both ends.
    _body = _text.split('\n', 1)[1].rsplit('\n', 1)[0]
    pathlib.Path(_name).write_text(_body)

print('wrote:', ', '.join(MODULES))

## 3. Mount Drive and authenticate

Two consent prompts: one to mount Drive, one for Google Cloud. Both are your own account — nothing is shared with anyone.

In [ ]:
from google.colab import auth, drive

drive.mount('/content/drive')
auth.authenticate_user()
print('Drive mounted and Google Cloud authenticated')

## 4. Preview the plan

Walks the mount and prints the tables it would build. **No BigQuery writes.**

`ROOT` is the folder to ingest. `/content/drive/MyDrive` is your whole Drive; narrow it to a subfolder to start smaller — e.g. `/content/drive/MyDrive/D:/Takeout` for just the health export.

In [ ]:
PROJECT = 'pelagic-gist-505800-b9'
ROOT = '/content/drive/MyDrive'
LOCATION = 'US'

!python pipeline.py plan --project $PROJECT --local-root "$ROOT"

## 5. Load

Review the table list above first. This is the step that writes to BigQuery.

Safe to re-run: file ids already marked loaded in the manifest are skipped, so an interrupted run resumes instead of duplicating rows. A file that fails is recorded in the manifest and does not abort the run.

Add `--dry-run` to log every operation without executing it. Add `--replace` to truncate the tables first rather than appending.

In [ ]:
!python pipeline.py load --project $PROJECT --local-root "$ROOT" \
    --location $LOCATION --from-cache

## 6. Check the result

Row counts per table, then anything that did not load and why.

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

print('--- rows per table ---')
tables = list(client.list_tables(f'{PROJECT}.drive_tables'))
for table in sorted(tables, key=lambda t: t.table_id):
    meta = client.get_table(table.reference)
    print(f'{table.table_id:<44} {meta.num_rows:>12,} rows'
          f'  {len(meta.schema):>3} cols')

print()
print('--- ingest status ---')
for row in client.query(f'''
    SELECT ingest_status, COUNT(*) AS files, SUM(row_count) AS rows
    FROM `{PROJECT}.drive_raw.file_manifest`
    GROUP BY ingest_status ORDER BY files DESC
'''):
    print(f'{row.ingest_status:<16} {row.files:>6} files  {row.rows or 0:>12,} rows')

### Anything that failed

Empty result means everything loaded.

In [ ]:
for row in client.query(f'''
    SELECT name, kind, fmt, size_bytes, ingest_error
    FROM `{PROJECT}.drive_raw.file_manifest`
    WHERE ingest_status = 'failed'
    ORDER BY size_bytes DESC LIMIT 50
'''):
    print(f'{row.name[:52]:<54} {row.ingest_error}')

### Example query

The point of the exercise — one query across the whole history that used to be hundreds of separate files.

In [ ]:
df = client.query(f'''
    SELECT _src_date AS day,
           COUNT(*) AS readings,
           ROUND(AVG(beats_per_minute), 1) AS avg_bpm,
           MIN(beats_per_minute) AS min_bpm,
           MAX(beats_per_minute) AS max_bpm
    FROM `{PROJECT}.drive_tables.heart_rate`
    WHERE beats_per_minute IS NOT NULL
    GROUP BY day ORDER BY day
''').to_dataframe()
df

---

**Note on native Google files.** Sheets, Docs and Slides are not real files on the mount — Drive represents them as small stub files with no data. The pipeline recovers their file id from the stub and exports them through the Drive API. If Colab's credentials lack the Drive scope they are recorded as `failed` in the manifest with a clear reason, and everything else still loads. Uploaded `.xlsx` files are real files and are unaffected.